In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica pelo método de Park (1999) aplicada à parte real da impedância
+ Split térmico sem overlap (sem vazamento)
+ Classificação multiclasse
+ Impressão COMPLETA de todas as métricas (ACC, F1, confusão, tempos)
+ Gráfico estilizado igual ao RF-Comp
+ Seleção por TEMP_DESEJADA
Autor: Luiz Eduardo Abdala José (adaptado)
"""

import re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning)


# ================================================================
# ===================== PARÂMETROS GERAIS ========================

ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30                    # temperatura alvo para referência

# >>>>>>>>>>>>>>>>>> ALTERE APENAS AQUI <<<<<<<<<<<<<<<<<<<<<<
TEMP_DESEJADA = 78           # temperatura da curva que será plotada
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50
SMOOTH_WIN = 5


# ------------------- PARK 1999 -------------------------
PARK_MAX_SHIFT_FRAC = 0.25
PARK_OVERLAP_MIN = 0.60
PARK_SMOOTH_WIN = 5
PARK_NSTEPS = 201

# ------------------- RANDOM FOREST ----------------------
RF_CLASSIF_PARAMS = dict(
    n_estimators=100,
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features=0.25,
    bootstrap=True,
    max_samples=0.70,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


# ================================================================
# ===================== FUNÇÕES AUXILIARES ======================
# ================================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c)
            freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs)[order]


def shift_interp(x, fhz, tau_hz):
    return np.interp(fhz, fhz + tau_hz, x, left=x[0], right=x[-1])


def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr.copy()
    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode='edge')
    kernel = np.ones(win)/win
    return np.convolve(arr_pad, kernel, mode="valid")[:len(arr)]


# ================================================================
# ========================= PARK 1999 ============================
# ================================================================

def park_compensate_single(x, y_ref, fhz,
                           max_shift_frac=PARK_MAX_SHIFT_FRAC,
                           overlap_min_frac=PARK_OVERLAP_MIN,
                           smooth_win=PARK_SMOOTH_WIN,
                           nsteps=PARK_NSTEPS):

    n = len(x)
    fmin, fmax = fhz[0], fhz[-1]
    df_band = fmax - fmin

    tau_max = max_shift_frac * df_band
    tau_vals = np.linspace(-tau_max, tau_max, nsteps)

    best = (np.inf, 0.0, 0.0)

    for tau in tau_vals:
        xs = shift_interp(x, fhz, tau)
        if len(xs) < int(overlap_min_frac * n):
            continue

        dS = float(np.mean(y_ref - xs))
        resid = (y_ref - (xs + dS))
        Va = float(np.sum(resid * resid))

        if Va < best[0]:
            best = (Va, tau, dS)

    _, tau_best, dS_best = best

    ycorr = shift_interp(x, fhz, tau_best) + dS_best
    if smooth_win > 1:
        ycorr = moving_average(ycorr, smooth_win)

    return ycorr, tau_best, dS_best


def park_batch(X, y_ref, fhz):
    Y = np.zeros_like(X)
    taus, deltas = [], []
    for i in range(len(X)):
        y, tau, dS = park_compensate_single(X[i], y_ref, fhz)
        Y[i] = y
        taus.append(tau)
        deltas.append(dS)
    return Y, np.array(taus), np.array(deltas)


# ================================================================
# ============================ SCRIPT =============================
# ================================================================

timings = {}

# ------------------ 1) CARREGA BASE ---------------------
t0 = time.time()
df = pd.read_pickle(ARQ_BASE)
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz_khz = fhz/1e3
timings["load"] = time.time() - t0
df_sem = df[df["falha"] == 0]

# ------------------ 2) SPLIT TÉRMICO ---------------------
t0 = time.time()
temps = sorted(df["temperatura_c"].unique())
temps_train = temps[::2]
temps_test = temps[1::2]

df_train_raw = df[df["temperatura_c"].isin(temps_train)].copy()
df_test_raw  = df[df["temperatura_c"].isin(temps_test)].copy()
timings["split"] = time.time() - t0

# ------------------ 3) REFERÊNCIA PARK -------------------
df_train_sem = df_train_raw[df_train_raw["falha"] == 0]
pool_ref = df_train_sem.loc[np.isclose(df_train_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)

y_ref = np.median(pool_ref, axis=0) if len(pool_ref)>0 else np.median(df_train_sem[fcols], axis=0)

# ------------------ 4) PARK TREINO -----------------------
t0 = time.time()
Y_train, _, _ = park_batch(df_train_raw[fcols].to_numpy(float), y_ref, fhz)
timings["park_train"] = time.time() - t0

# ------------------ 5) PARK TESTE ------------------------
t0 = time.time()
Y_test, _, _ = park_batch(df_test_raw[fcols].to_numpy(float), y_ref, fhz)
timings["park_test"] = time.time() - t0

X_train = Y_train
X_test = Y_test
y_train = df_train_raw["falha"].to_numpy(int)
y_test = df_test_raw["falha"].to_numpy(int)

# ------------------ 6) CLASSIFICAÇÃO ---------------------
t0 = time.time()
clf = RandomForestClassifier(**RF_CLASSIF_PARAMS).fit(X_train, y_train)
timings["train_clf"] = time.time() - t0

t0 = time.time()
y_pred = clf.predict(X_test)
timings["predict_clf"] = time.time() - t0

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
cm = confusion_matrix(y_test, y_pred)

# ------------------ 7) ESCOLHE CURVA PARA PLOTAR ---------
df_plot = df_test_raw[df_test_raw["temperatura_c"] == TEMP_DESEJADA]

if len(df_plot) == 0:
    raise ValueError(f"NÃO existe curva de temperatura {TEMP_DESEJADA}°C no conjunto de teste!")

idx_show = df_plot.index[0]

y_orig = df.loc[idx_show, fcols].to_numpy(float)
y_comp = Y_test[df_test_raw.index.get_loc(idx_show)]

# ================================================================
# =========================== PRINTS ==============================
# ================================================================

print("\n============== CLASSIFICAÇÃO (Park) ===================")
print("Matriz de confusão:")
print(cm)
print(f"\nACC = {acc:.4f}")
print(f"F1  = {macro_f1:.4f}")
print("\nRelatório completo:")
print(classification_report(y_test, y_pred, digits=4))

print("\n============== TEMPOS (s) ==================")
for k,v in timings.items():
    print(f"{k:20s}: {v:.4f}")


# ================================================================
# =========================== GRÁFICO =============================
# ================================================================

print(f"\n🔹 Plotando curva a {TEMP_DESEJADA}°C (Park) ...")

plt.rcParams.update({
    'font.size': 13,
    'text.usetex': False,
    'font.family': "Times New Roman"
})

plt.figure(figsize=(12,6))
plt.plot(fhz_khz, y_ref, '--', c='black', lw=1.2, label=f"Referência {REF_TEMP}°C")
plt.plot(fhz_khz, y_orig, c='tab:red', lw=1.5, alpha=0.6,
         label=f"Original {TEMP_DESEJADA}°C ")
plt.plot(fhz_khz, y_comp, c='tab:blue', lw=2,
         label=f"Compensado Park {TEMP_DESEJADA}°C")

plt.title(f"Compensação Park — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.grid(alpha=0)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ================================================================
# =========================== GRÁFICO =============================
# ================================================================

print(f"\n🔹 Plotando curva a {TEMP_DESEJADA}°C (Park) ...")

plt.rcParams.update({
    'font.size': 10,
    'text.usetex': False,
    'font.family': "Times New Roman"
})

plt.figure(figsize=(8,4))
plt.plot(fhz_khz, y_ref, '--', c='black', lw=1.2, label=f"Referência {REF_TEMP}°C")
plt.plot(fhz_khz, y_orig, c='tab:red', lw=1.5, alpha=0.6,
         label=f"Original {TEMP_DESEJADA}°C ")
plt.plot(fhz_khz, y_comp, c='tab:blue', lw=2,
         label=f"Compensado Park {TEMP_DESEJADA}°C")

plt.title(f"Compensação Park — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.grid(alpha=0)
plt.legend(frameon=True,
           facecolor='white',
           edgecolor='none')
plt.tight_layout()
plt.show()
